#Baseline BERT

In [1]:
import pandas as pd
import numpy as np
# import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from imblearn import over_sampling
from datasets import Dataset
from IPython.display import display
from transformers import BertForSequenceClassification, BertConfig, BertTokenizer
from transformers import Trainer, TrainingArguments

# Reading product review sentiment file
df_prod_review = pd.read_csv('https://raw.githubusercontent.com/aakashgoel12/blogs/master/input/product_review_sentiment.csv',\
                      encoding='latin-1')
display(df_prod_review.sample(n=5, random_state=42))

,Review,user_sentiment
9329,fresh clean smell everything need quick clean ...,1
4160,great vacuum love lightweight vacuum easy carr...,1
18500,smell great wipe easy use work smell great,1
8840,product count use clorox wipe everything trave...,1
5098,great movie excellent movie add blu ray collec...,1


In [2]:
x=df_prod_review['Review']
y=df_prod_review['user_sentiment']
print("Checking distribution of +ve and -ve review sentiment: \n{}".format(y.value_counts(normalize=True)))
# Split the dataset into test and train
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=50)

# As we saw above that data is imbalanced, balance training data using over sampling

ros = over_sampling.RandomOverSampler(random_state=0)
X_train, y_train = ros.fit_resample(pd.DataFrame(X_train), pd.Series(y_train))
print("Checking distribution of +ve and -ve review sentiment after oversampling: \n{}".format(y_train.value_counts(normalize=True)))
# Convert into list of string
X_train = X_train['Review'].tolist()

Checking distribution of +ve and -ve review sentiment: 
user_sentiment
1    0.888401
0    0.111599
Name: proportion, dtype: float64
Checking distribution of +ve and -ve review sentiment after oversampling: 
user_sentiment
1    0.5
0    0.5
Name: proportion, dtype: float64


In [3]:
# Make Huggingface Dataset
train_data = Dataset.from_dict({"text": X_train, "label": y_train})
test_data = Dataset.from_dict({"text": X_test, "label": y_test})

In [4]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

train_tokenized = train_data.map(tokenize, batched=True)
test_tokenized = test_data.map(tokenize, batched=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/33468 [00:00<?, ? examples/s]

Map:   0%|          | 0/8062 [00:00<?, ? examples/s]

In [6]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
training_args = TrainingArguments(
    output_dir='./results', # All files generated during training will be stored here
    num_train_epochs=3, # The model will be trained for 3 full epochs unless the step limit (max_steps) is reached first
    per_device_train_batch_size=16, # Training batch size per device (GPU or CPU).
    per_device_eval_batch_size=64, # Evaluation batch size per device (GPU or CPU).
    weight_decay=0.01, # Weight decay rate: this technique helps to avoid overfitting, penalizing large weights in the neural network
    logging_dir='./logs', # Directory where training logs will be stored
    eval_strategy="epoch",
    save_strategy="epoch",  # Range of steps after which the model will be saved
    logging_steps=10,  # Range of steps after which log information will be recorded
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [8]:
trainer = Trainer(
    model=model, # The pre-trained model that you want to fine-tune or train
    args=training_args, # The training arguments that specify the configurations for the training process
    train_dataset=train_tokenized, # The dataset used for training the model
    eval_dataset=test_tokenized # The dataset used for evaluating the model during training
)

# Start training
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.110221,0.257105
2,0.038576,0.341903
3,0.001380,0.363241


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6276, training_loss=0.12300187723080161, metrics={'train_runtime': 2365.5565, 'train_samples_per_second': 42.444, 'train_steps_per_second': 2.653, 'total_flos': 6604350600591360.0, 'train_loss': 0.12300187723080161, 'epoch': 3.0})

In [9]:
preds_output = trainer.predict(test_tokenized)
preds = preds_output.predictions.argmax(-1)

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.80      0.61      0.69       922
           1       0.95      0.98      0.97      7140

    accuracy                           0.94      8062
   macro avg       0.88      0.80      0.83      8062
weighted avg       0.93      0.94      0.93      8062



##Error Analysis

In [10]:
true_labels = y_test.tolist()
pred_labels = preds.tolist()
texts = X_test.tolist()

error_df = pd.DataFrame({
    'text': texts,
    'actual': true_labels,
    'predicted': pred_labels
})

In [ ]:
error_df.to_csv('prediksi vs actual BERT.csv', index=False)

In [11]:
# Inspect misclassified samples
misclassified = error_df[error_df['actual'] != error_df['predicted']]

print("Jumlah total error:", len(misclassified))
display(misclassified.sample(10, random_state=42))

misclassified.to_csv('misclassified_bert_baseline.csv', index=False)

Jumlah total error: 498


,text,actual,predicted
7909,awful product first consistency good previous ...,0,1
1036,great movie godzilla fan always big fan godzil...,0,1
3615,excellent movie show fantastic collection resi...,0,1
2508,moisturize tube lip shimmer like fact put stay...,0,1
3724,mess lotion watery separate arrived ive use pr...,0,1
6867,one best ive use product several year still us...,0,1
2283,grandson love godzilla movie grandson watch mo...,0,1
730,roar get steelbook version sound sometimes acc...,1,0
5132,good lip balm lipstick moisturizing like beewa...,0,1
120,first grade class use wipe daughter st grade ...,0,1


##Sampling

In [12]:
sampled_df = error_df.sample(n=300, random_state=42)
display(sampled_df)

,text,actual,predicted
742,awsome product really good cleaning defect rea...,1,1
2126,refresh year old daughter love lip shimmer ...,1,1
3689,great story didnt purchase self however person...,1,1
4586,worth every cent product best moisturizer lips...,1,1
6289,good movie good,1,1
...,...,...,...
5019,change good thing use exxtra gel long remember...,0,0
7450,clorox wipe love clorox product especially wip...,1,1
1606,great product smooth go easily immediately abs...,1,1
1405,great product great product use daily housethe...,1,1


In [13]:
# Inspect misclassified samples
misclassified_sample = sampled_df[sampled_df['actual'] != sampled_df['predicted']]

print("Jumlah total error:", len(misclassified_sample))
display(misclassified_sample.sample(10, random_state=42))

Jumlah total error: 13


,text,actual,predicted
1992,best wipe love smell cant go wrong clorox prod...,0,1
4769,great collection resident evil movie one low p...,0,1
7886,great organizational product folder assist org...,0,1
2283,grandson love godzilla movie grandson watch mo...,0,1
487,hilarious full non stop laugh one funniest rau...,1,0
1452,surprised vacuum clean read review hesitant pu...,0,1
3151,well well big brother cheetos cheese puff bite...,1,0
730,roar get steelbook version sound sometimes acc...,1,0
7816,lemon fresh recently use clorox disinfect wipe...,1,0
4148,omg nothing work well impacted wisdom tooth we...,1,0


In [14]:
# # Create a copy before adding baseline prediction labels
sampled_with_acc = sampled_df.copy()

# acc_b: 1 = correct prediction, 0 = misclassified
sampled_with_acc['acc_b'] = sampled_with_acc['text'].apply(
    lambda x: 0 if x in misclassified_sample['text'].values else 1
)

display(sampled_with_acc)

,text,actual,predicted,acc_b
742,awsome product really good cleaning defect rea...,1,1,1
2126,refresh year old daughter love lip shimmer ...,1,1,1
3689,great story didnt purchase self however person...,1,1,1
4586,worth every cent product best moisturizer lips...,1,1,1
6289,good movie good,1,1,1
...,...,...,...,...
5019,change good thing use exxtra gel long remember...,0,0,1
7450,clorox wipe love clorox product especially wip...,1,1,1
1606,great product smooth go easily immediately abs...,1,1,1
1405,great product great product use daily housethe...,1,1,1


In [ ]:
sampled_with_acc.to_csv('BERT_baseline_sampel.csv', index=False)